In [1]:

# ============================================================
# 0. IMPORT & INSTALL
# ============================================================
!pip install imbalanced-learn xgboost --quiet

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42


In [2]:

# ============================================================
# 1. LOAD DATASET
# ============================================================
df = pd.read_csv('ifls5_heart_attack_dataset.csv')

X = df.drop(columns=['pidlink', 'HeartAttack'])
y = df['HeartAttack']

print("Jumlah data:", X.shape)
print("Distribusi target:\n", y.value_counts())


Jumlah data: (32173, 10)
Distribusi target:
 HeartAttack
0.0    31858
1.0      315
Name: count, dtype: int64


In [3]:

# ============================================================
# 2. SPLIT TRAIN-TEST (stratified, karena target imbalance)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("\nTrain:", y_train.value_counts().to_dict())
print("Test :", y_test.value_counts().to_dict())



Train: {0.0: 25486, 1.0: 252}
Test : {0.0: 6372, 1.0: 63}


In [4]:

# ============================================================
# 3. PREPROCESSING (imputasi + scaling)
# ============================================================
numeric_cols = X.columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols)
])

preprocessor.fit(X_train)                       # fit HANYA di train
X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)


In [5]:

# ============================================================
# 4. FEATURE SELECTION PAKAI XGBOOST FEATURE IMPORTANCE
# ============================================================
ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb_selector = XGBClassifier(
    n_estimators=300, random_state=RANDOM_STATE, scale_pos_weight=ratio, eval_metric='logloss'
)
xgb_selector.fit(X_train_processed, y_train)

importances = pd.Series(xgb_selector.feature_importances_, index=numeric_cols).sort_values(ascending=False)
print("\nUrutan feature importance (XGBoost):\n", importances)

# Keputusan final: pakai 4 fitur teratas (sudah dibandingkan vs 8 fitur, hasilnya lebih stabil)
selected_features = importances.head(4).index.tolist()
print("\nFitur terpilih (final):", selected_features)

selected_idx = [numeric_cols.index(f) for f in selected_features]
X_train_sel = X_train_processed[:, selected_idx]
X_test_sel  = X_test_processed[:, selected_idx]



Urutan feature importance (XGBoost):
 Diabetes           0.140564
HighCholesterol    0.114018
BMI                0.107959
Weight_kg          0.103209
DiastolicBP        0.096582
Age                0.096130
Height_cm          0.091223
SystolicBP         0.089529
Hypertension       0.088508
Sex                0.072277
dtype: float32

Fitur terpilih (final): ['Diabetes', 'HighCholesterol', 'BMI', 'Weight_kg']


In [6]:

# ============================================================
# 5. SMOTE (HANYA di data train, target imbalance ~1%)
# ============================================================
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_sel, y_train)

print("\nSebelum SMOTE:", y_train.value_counts().to_dict())
print("Sesudah SMOTE:", pd.Series(y_train_res).value_counts().to_dict())



Sebelum SMOTE: {0.0: 25486, 1.0: 252}
Sesudah SMOTE: {0.0: 25486, 1.0: 25486}


In [7]:

# ============================================================
# 6. DEFINISI MODEL (6 model + FDA)
# ============================================================
# FDA: Python tidak punya library FDA resmi. Pendekatan yang dipakai di sini adalah
# aproksimasi FDA lewat "basis expansion" (PolynomialFeatures, derajat 2) + LDA —
# ini WAJIB disebutkan sebagai pendekatan/aproksimasi di metodologi paper, bukan
# implementasi FDA murni seperti library `mda` di R.
fda_approx = Pipeline([
    ('basis_expansion', PolynomialFeatures(degree=2, include_bias=False)),
    ('lda', LinearDiscriminantAnalysis())
])

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=300, random_state=RANDOM_STATE),
    'SVM': SVC(class_weight='balanced', probability=True, random_state=RANDOM_STATE),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=RANDOM_STATE),
    'FDA (basis expansion + LDA)': fda_approx
}


In [8]:

# ============================================================
# 7. 10-FOLD STRATIFIED CROSS-VALIDATION (di data train hasil SMOTE)
# ============================================================
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print("\n=== HASIL CROSS-VALIDATION (10-fold) ===")
cv_results = {}
for name, model in models.items():
    scores = cross_validate(model, X_train_res, y_train_res, cv=cv, scoring=scoring)
    cv_results[name] = {m: scores[f'test_{m}'].mean() for m in scoring}
    print(name, cv_results[name])

cv_results_df = pd.DataFrame(cv_results).T



=== HASIL CROSS-VALIDATION (10-fold) ===
Logistic Regression {'accuracy': np.float64(0.591403088440473), 'precision': np.float64(0.6089653399530098), 'recall': np.float64(0.510984869247213), 'f1': np.float64(0.5556344283323693), 'roc_auc': np.float64(0.6310196870356546)}
KNN {'accuracy': np.float64(0.8439731738598379), 'precision': np.float64(0.8005213370803895), 'recall': np.float64(0.9163460845605103), 'f1': np.float64(0.8545170550223655), 'roc_auc': np.float64(0.9082177506017001)}
Decision Tree {'accuracy': np.float64(0.8778937494520773), 'precision': np.float64(0.8791335741135434), 'recall': np.float64(0.8764021104714935), 'f1': np.float64(0.8777201741180856), 'roc_auc': np.float64(0.8785190912878468)}
Random Forest {'accuracy': np.float64(0.8943144387659322), 'precision': np.float64(0.8817653185797656), 'recall': np.float64(0.9108528877948258), 'f1': np.float64(0.8960501796837699), 'roc_auc': np.float64(0.9608242942370582)}
SVM {'accuracy': np.float64(0.618005853180353), 'precisi

In [9]:

# ============================================================
# 8. EVALUASI FINAL DI TEST SET (data asli, TIDAK di-SMOTE)
# ============================================================
print("\n=== HASIL EVALUASI FINAL (TEST SET) ===")
final_results = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_sel)
    y_proba = model.predict_proba(X_test_sel)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test_sel)

    final_results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }
    print(f"\n=== {name} ===")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))

results_df = pd.DataFrame(final_results).T
print("\n=== RINGKASAN SEMUA MODEL ===")
print(results_df.sort_values('ROC-AUC', ascending=False))



=== HASIL EVALUASI FINAL (TEST SET) ===

=== Logistic Regression ===
[[4266 2106]
 [  28   35]]
              precision    recall  f1-score   support

         0.0       0.99      0.67      0.80      6372
         1.0       0.02      0.56      0.03        63

    accuracy                           0.67      6435
   macro avg       0.50      0.61      0.42      6435
weighted avg       0.98      0.67      0.79      6435


=== KNN ===
[[4925 1447]
 [  45   18]]
              precision    recall  f1-score   support

         0.0       0.99      0.77      0.87      6372
         1.0       0.01      0.29      0.02        63

    accuracy                           0.77      6435
   macro avg       0.50      0.53      0.45      6435
weighted avg       0.98      0.77      0.86      6435


=== Decision Tree ===
[[5603  769]
 [  54    9]]
              precision    recall  f1-score   support

         0.0       0.99      0.88      0.93      6372
         1.0       0.01      0.14      0.02       